This notebooks includes some experiments included in Section DSNU based source camera identification of the article. 

This code provides the following results:
1. DSNU in raw format using the full image. 
1. DSNU in raw format splitting channels.

## DSNU in raw format using the full image

In [ ]:
import os
import numpy as np
import rawpy
from glob import glob

In [ ]:
def open_image(img_path):
    with rawpy.imread(img_path) as img:
        return img.raw_image_visible.astype(np.float32)

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

raw_exts = ['cr2', 'cr3', 'arw']

images = []
devices = []

for tag in tags:
    dark_dir = os.path.join(base_dir, tag, 'dark/raw')
    
    for ext in raw_exts:
        for img_path in glob(os.path.join(dark_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            images.append(img_path)
            devices.append(device)

images = np.array(images)
devices = np.array(devices)

In [ ]:
print('Computing DSNU fingerprints')

crop_size = (2823, 4240)

fingerprint_devices = sorted(np.unique(devices))
k = []
evaluation_sets = {}

for device in fingerprint_devices:
    
    device_imgs = images[devices == device]
    
    dsnu_paths = device_imgs[:50]
    eval_paths = device_imgs[50:]
    
    evaluation_sets[device] = eval_paths
    
    dsnu_imgs = []
    
    for path in dsnu_paths:
        im = open_image(path)
        im = cut_fixed(im, crop_size)
        
        if im is None:
            continue
        
        dsnu_imgs.append(im)
    
    if len(dsnu_imgs) == 0:
        continue
    
    dsnu_imgs = np.stack(dsnu_imgs, axis=0)
    
    fingerprint = np.mean(dsnu_imgs, axis=0)
    
    
    k.append(fingerprint)

k = np.stack(k, axis=0)

In [ ]:
eval_items = []

for device in fingerprint_devices:
    for path in evaluation_sets[device]:
        eval_items.append({
            'path': path,
            'device': device
        })

In [ ]:
matching_cases = []
mismatching_cases = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]

    print('Processing fingerprint:', device_fp)

    for item in eval_items:
        path = item['path']
        device_nat = item['device']

        im = open_image(path)
        im = cut_fixed(im, crop_size)

        if im is None:
            continue


        cc2d = prnu.crosscorr_2d(fingerprint_k, im)
        pce_value = prnu.pce(cc2d)['pce']

        case = {
            'pce': float(pce_value),
            'fingerprint_device': device_fp,
            'natural_device': device_nat,
            'natural_path': path
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    "pce_dsnu-dark.npz",
    matching_cases=matching_cases,
    mismatching_cases=mismatching_cases
)

In [ ]:
data = np.load("pce_dsnu-dark.npz", allow_pickle=True)
matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

from collections import defaultdict

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pce'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pce'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]

plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)

plt.xticks(range(len(sensors)), sensors, rotation=45)
plt.yscale('log')
plt.xlabel("Sensor", fontsize =16)
plt.ylabel("sPCE value", fontsize = 16)
plt.title("sPCE distribution per sensor DSNU using dark frames", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import rawpy
from glob import glob

from scipy.stats import pearsonr

In [ ]:
def open_image(img_path):
    with rawpy.imread(img_path) as img:
        return img.raw_image_visible.astype(np.float32)

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

raw_exts = ['cr2', 'cr3', 'arw']

images = []
devices = []

for tag in tags:
    dark_dir = os.path.join(base_dir, tag, 'dark/raw')
    
    for ext in raw_exts:
        for img_path in glob(os.path.join(dark_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            images.append(img_path)
            devices.append(device)

images = np.array(images)
devices = np.array(devices)

In [ ]:
print('Computing DSNU fingerprints')

crop_size = (2823, 4240)

fingerprint_devices = sorted(np.unique(devices))
k = []
evaluation_sets = {}

for device in fingerprint_devices:
    
    device_imgs = images[devices == device]
    
    dsnu_paths = device_imgs[:50]
    eval_paths = device_imgs[50:]
    
    evaluation_sets[device] = eval_paths
    
    dsnu_imgs = []
    
    for path in dsnu_paths:
        im = open_image(path)
        im = cut_fixed(im, crop_size)
        
        if im is None:
            continue
        
        dsnu_imgs.append(im)
    
    if len(dsnu_imgs) == 0:
        continue
    
    dsnu_imgs = np.stack(dsnu_imgs, axis=0)
    
    fingerprint = np.mean(dsnu_imgs, axis=0)
    
    
    k.append(fingerprint)

k = np.stack(k, axis=0)

In [ ]:
eval_items = []

for device in fingerprint_devices:
    for path in evaluation_sets[device]:
        eval_items.append({
            'path': path,
            'device': device
        })

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
matching_cases = []
mismatching_cases = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]

    print('Processing fingerprint:', device_fp)

    for item in eval_items:
        path = item['path']
        device_nat = item['device']

        im = open_image(path)
        im = cut_fixed(im, crop_size)

        if im is None:
            continue

        pearson = pearson_corr(fingerprint_k, im)

        case = {
            'pearson': float(pearson),
            'fingerprint_device': device_fp,
            'natural_device': device_nat,
            'natural_path': path
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    "pearson_dsnu-dark.npz",
    matching_cases=matching_cases,
    mismatching_cases=mismatching_cases
)

In [ ]:
data = np.load("pearson_dsnu-dark.npz", allow_pickle=True)
matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

from collections import defaultdict

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pearson'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pearson'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]

import matplotlib.pyplot as plt
plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)


plt.xticks(range(len(sensors)), sensors, rotation=45)
plt.xlabel("Sensor", fontsize =16)
plt.ylabel("Pearson correlation value", fontsize = 16)
plt.title("Pearson correlation distribution per sensor DSNU with dark frames", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

## DSNU in raw format splitting by channel

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import matplotlib.pyplot as plt

import rawpy

from collections import Counter

In [ ]:
def split_bayer(im):
    R  = im[0::2, 0::2]
    G1 = im[0::2, 1::2]
    G2 = im[1::2, 0::2]
    B  = im[1::2, 1::2]
    return [R, G1, G2, B]

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape[:2]
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]


def enforce_even(img):
    H, W = img.shape
    return img[:H - (H % 2), :W - (W % 2)]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

dark_images, dark_devices = [], []

raw_exts = ['cr3', 'cr2', 'arw']

for tag in tags:
    dark_dir = os.path.join(base_dir, tag, 'dark', 'raw')
    
    for ext in raw_exts:
        for img_path in glob(os.path.join(dark_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            dark_images.append(img_path)
            dark_devices.append(device)

dark_images = np.array(dark_images)
dark_devices = np.array(dark_devices)

In [ ]:
def open_image(img_path):
    img = rawpy.imread(img_path)
    return img.raw_image_visible

In [ ]:
crop_size = (2848, 4256)
print('Computing DSNU (first 50 darks per device)')

fingerprint_devices = sorted(np.unique(dark_devices))
k = []

test_images = []
test_devices = []

for device in fingerprint_devices:
    print("Device:", device)
    
    device_imgs = dark_images[dark_devices == device]
    
    if len(device_imgs) < 51:
        raise ValueError(f"Device {device} needs at least 51 dark frames")
    
    fp_imgs = device_imgs[:50]
    test_imgs = device_imgs[50:]
    
    test_images.extend(test_imgs)
    test_devices.extend([device] * len(test_imgs))
    
    channel_imgs = [[], [], [], []]  # R, G1, G2, B
    
    for img_path in fp_imgs:
        im = np.asarray(open_image(img_path)).astype(np.float32)
        
        im = enforce_even(im)
        im_crop = cut_fixed(im, crop_size)
        
        if im_crop is None:
            raise ValueError(f"Crop failed in {img_path}")
        
        channels = split_bayer(im_crop)
        
        for c in range(4):
            channel_imgs[c].append(channels[c])
    
    k_device = []
    for c in range(4):
        imgs_c = np.stack(channel_imgs[c], axis=0)
        k_c = np.mean(imgs_c, axis=0)
        k_device.append(k_c)
    
    k.append(k_device)

k = np.array(k)
print("k shape:", k.shape)

test_images = np.array(test_images)
test_devices = np.array(test_devices)

In [ ]:
from scipy.stats import pearsonr
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
output_file = 'spce-pearson-dark-dsnu-split.npz'

if os.path.exists(output_file):
    data = np.load(output_file, allow_pickle=True)
    results = list(data['results'])
    
    processed_imgs = {r['img_idx'] for r in results}
    start_idx = max(processed_imgs) + 1 if processed_imgs else 0
    
    print(f"Restarting from image {start_idx}")
else:
    results = []
    start_idx = 0

print('Processing test dark images')

for n_idx in range(start_idx, len(test_images)):
    img_path = test_images[n_idx]
    device_nat = test_devices[n_idx]
    
    print(f"Image {n_idx}")

    im = np.asarray(open_image(img_path)).astype(np.float32)
    
    im = enforce_even(im)
    im_crop = cut_fixed(im, crop_size)
    
    if im_crop is None:
        raise ValueError(f"Crop failed in {img_path}")
    
    channels = split_bayer(im_crop)

    w_channels = []
    for c in range(4):
        w_c = prnu_raw.extract_single(channels[c])
        w_channels.append(w_c)


    for f_idx, fingerprint_k in enumerate(k):
        device_fp = fingerprint_devices[f_idx]
        
        for c in range(4):
            cc2d = prnu_raw.crosscorr_2d(fingerprint_k[c], w_channels[c])
            pce_value = prnu_raw.pce(cc2d)['pce']
            
            pearson = pearson_corr(fingerprint_k[c], w_channels[c])
            
            results.append({
                'pce': pce_value,
                'pearson': pearson,
                'match': int(device_fp == device_nat),
                'fp_device': device_fp,
                'nat_device': device_nat,
                'img_idx': n_idx,
                'channel': c
            })

    if n_idx % 5 == 0:
        print("Savin checkpoint...")
        np.savez(output_file, results=results)

np.savez(output_file, results=results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import os

channel_map = {
    0: 'B',
    1: 'G1',
    2: 'G2',
    3: 'R'
}

output_dir = "spce-pearson-dark-dsnu-split"
os.makedirs(output_dir, exist_ok=True)

data = np.load('spce-pearson-dark-dsnu-split.npz', allow_pickle=True)
results = data['results']

channel_names = sorted(list(set(channel_map[case['channel']] for case in results)))

for cname in channel_names:

    sensor_data_pce = defaultdict(lambda: {'match': [], 'mismatch': []})
    sensor_data_pearson = defaultdict(lambda: {'match': [], 'mismatch': []})

    for case in results:
        if channel_map[case['channel']] != cname:
            continue

        sensor = case['fp_device']
        
        if case['match']:
            sensor_data_pce[sensor]['match'].append(case['pce'])
            sensor_data_pearson[sensor]['match'].append(case['pearson'])
        else:
            sensor_data_pce[sensor]['mismatch'].append(case['pce'])
            sensor_data_pearson[sensor]['mismatch'].append(case['pearson'])

    sensors = sorted(sensor_data_pce.keys())

    positions_match = [i - 0.15 for i in range(len(sensors))]
    positions_mismatch = [i + 0.15 for i in range(len(sensors))]

    match_values = [sensor_data_pce[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pce[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.yscale('log')
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("sPCE value", fontsize=16)
    plt.title(f"sPCE distribution per sensor - Channel {cname} DSNU", fontsize=16)

    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pce_{cname}.png"), dpi=300)
    plt.close()

    match_values = [sensor_data_pearson[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pearson[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("Pearson correlation", fontsize=16)
    plt.title(f"Pearson distribution per sensor - Channel {cname} DSNU", fontsize=16)

    plt.grid(True, linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pearson_{cname}.png"), dpi=300)
    plt.close()